In [1]:
from datasets import load_dataset
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import re

In [ ]:
#carica direttamente il JSON aggirando i problemi di sicurezza di HF
APPS = load_dataset(
    "json", 
    data_files={
        "train": "hf://datasets/codeparrot/apps/train.jsonl",
        "test": "hf://datasets/codeparrot/apps/test.jsonl"
    }
)

In [3]:
train_data = APPS['train']
test_data = APPS['test']

In [4]:
print(train_data.features)
print(test_data.features)

{'id': Value(dtype='int64', id=None), 'question': Value(dtype='string', id=None), 'solutions': Value(dtype='string', id=None), 'input_output': Value(dtype='string', id=None), 'difficulty': Value(dtype='string', id=None), 'url': Value(dtype='string', id=None), 'starter_code': Value(dtype='string', id=None)}
{'id': Value(dtype='int64', id=None), 'question': Value(dtype='string', id=None), 'solutions': Value(dtype='string', id=None), 'input_output': Value(dtype='string', id=None), 'difficulty': Value(dtype='string', id=None), 'url': Value(dtype='string', id=None), 'starter_code': Value(dtype='string', id=None)}


In [5]:
train_question = train_data['question']

In [6]:
question = train_question[0]

In [7]:
template = ChatPromptTemplate(
    [
        ('system', 'You are a helpful assistant for coding, return only the solution for the coding problem (no test cases nedded, no exammple usage nedded) described by the user in python,'
        'put | before and after the script'),
        ('human', '{user_prompt}')
    ]
)


In [8]:
# change this line to evaluate different models
llm = ChatOllama(model='mistral-nemo')
chain = template | llm | StrOutputParser()  # FIX 1: aggiunto ()

response = chain.invoke({"user_prompt": question})
print(response)  # FIX 2: StrOutputParser() ritorna già una stringa, tolto .content

C:\Users\dernj\AppData\Local\Temp\ipykernel_32916\1601159057.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model='mistral-nemo')


|def solve(n, words):
    graph = [[] for _ in range(2)]
    for i, word in enumerate(words):
        graph[ord(word[-1])].append(i)
    visited = [False] * n
    def dfs(node, parent):
        if visited[node]:
            return False
        visited[node] = True
        for child in graph[node]:
            if child != parent and not dfs(child, node):
                return False
        return True
    ans = []
    for i in range(n):
        if not dfs(i, -1):
            ans.append(i)
    return len(ans), ans|


In [9]:
template_resampling = ChatPromptTemplate(
    [
        ('system', 
         'You are a prompt resampler. Given a competitive programming problem, generate EXACTLY 5 '
         'complete rewordings of the ENTIRE problem. '
         'Each rewriting must:\n'
         '- Contain ALL the original information (constraints, input format, output format, examples)\n'
         '- Be a COMPLETE standalone problem description\n'
         '- Use different wording and sentence structure\n\n'
         'FORMAT (mandatory):\n'
         '|complete reworded problem 1|\n'
         '|complete reworded problem 2|\n'
         '...\n'
         'Output NOTHING outside the | | delimiters.'),
        ('human', '{user_prompt}')
    ]
)
llm_resampling = ChatOllama(model='llama3.1:8b')
chain_res = template_resampling | llm_resampling | StrOutputParser()
response_resampled = chain_res.invoke({'user_prompt': question})

In [10]:
resampled_list = response_resampled.split('|')
resampled_list = [
    item.strip() 
    for item in resampled_list 
    if item.strip() and item.strip() != '\n\n'
]

for i, variant in enumerate(resampled_list, 1):
    print(f"{i}. {variant}")

1. For a given set of n different binary words, Polycarp wants to play a game "words" where the next word must start with the last character of the previous word. However, it's possible that there is no way to arrange these words according to the game rules. In such cases, he wants to reverse some words from his set so that the final sequence of n words is consistent with the game rules and all words remain unique. Polycarp needs our help in finding the minimum number of words that need to be reversed.

Input:
The input consists of t test cases. Each test case starts with a line containing the integer n, which represents the number of words in the set. The next n lines contain these binary words. All words are different and not empty, consisting only of characters '0' and '1'.

Output:
For each test case, we need to print the minimum number of words that should be reversed, followed by their indices (from 1 to n) if it's a non-zero value.
2. Given a collection of distinct binary string

In [11]:
def extract_code(response: str) -> str:
    """Estrae il codice dalla risposta del modello."""
    # Prova prima con i pipe | ... |
    pipe_match = re.findall(r'\|(.+?)\|', response, re.DOTALL)
    if pipe_match:
        return max(pipe_match, key=len).strip()  # prendi il blocco più lungo
    
    # Fallback: prova con i blocchi ```python ... ```
    code_match = re.findall(r'```python(.+?)```', response, re.DOTALL)
    if code_match:
        return max(code_match, key=len).strip()
    
    # Fallback finale: restituisci tutto
    return response.strip()

In [12]:
variant_response = []
for variant in resampled_list:
    raw_response = chain.invoke({"user_prompt": variant})
    code = extract_code(raw_response)
    variant_response.append({
        "variant": variant,
        "raw": raw_response,
        "code": code
    })

In [13]:
print(variant_response)

[{'variant': 'For a given set of n different binary words, Polycarp wants to play a game "words" where the next word must start with the last character of the previous word. However, it\'s possible that there is no way to arrange these words according to the game rules. In such cases, he wants to reverse some words from his set so that the final sequence of n words is consistent with the game rules and all words remain unique. Polycarp needs our help in finding the minimum number of words that need to be reversed.\n\nInput:\nThe input consists of t test cases. Each test case starts with a line containing the integer n, which represents the number of words in the set. The next n lines contain these binary words. All words are different and not empty, consisting only of characters \'0\' and \'1\'.\n\nOutput:\nFor each test case, we need to print the minimum number of words that should be reversed, followed by their indices (from 1 to n) if it\'s a non-zero value.', 'raw': '|def solve(n, 